# E791 $D^+\to\pi^-\pi^+\pi^+$ — GenFit bias study

Repeated coefficient-fit closure test using the native `GenFit` implementation. This validation run uses **100 pseudoexperiments × 50,000 events**. Resonance masses, widths, spins and meson radii are fixed; the $\rho(770)\pi^+$ coefficient is fixed to $1+0i$.

The primary goal is a fit-bias test under independent finite-sample statistical fluctuations. Following the basin-of-attraction study, each fit starts from a Gaussian perturbation around the injected truth with $\sigma_{start}=0.25$. This width recovered the physical minimum in 20/20 starts in the single-toy diagnostic, whereas wider starts increasingly populated local minima.

Numerical convergence and fit-quality acceptance are handled first; the bias and pull-calibration table then uses all accepted fits without distribution-level outlier rejection. Robust median/MAD views are secondary diagnostics only.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from dalitzplotfitter import (
    DecayChannel, DecayModel, GenFit, NonResonant, Parameter,
    RealImag, Resonance, enable_x64, genfit_bias_summary,
    genfit_distribution, genfit_outlier_selection,
    genfit_robust_gaussian_fit, genfit_robust_summary,
    print_genfit_bias_summary, print_genfit_robust_summary,
    robust_outlier_mask, robust_gaussian_fit,
)
enable_x64()


## 1. E791 Fit-2 model


In [ ]:
channel = DecayChannel("D+", ("pi-", "pi+", "pi+"))
fit2_polar = {
    "sigma": (1.17,205.7), "rho770": (1.00,0.0), "NR": (0.48,57.3),
    "f0_980": (0.43,165.0), "f2_1270": (0.76,57.3),
    "f0_1370": (0.26,105.4), "rho1450": (0.14,319.1),
}
def polar_to_xy(r, phase_deg):
    phase=np.deg2rad(phase_deg); return r*np.cos(phase), r*np.sin(phase)
def internal_xy(name):
    r,phase=fit2_polar[name]
    if name=="NR": phase += 180.0
    return polar_to_xy(r,phase)
truth_xy={name:internal_xy(name) for name in fit2_polar}


In [ ]:
truth={}
START_SIGMA=0.25
def free_coefficient(name):
    x,y=truth_xy[name]
    truth[f"{name}.x"]=float(x); truth[f"{name}.y"]=float(y)
    # GenFit with start_range=None draws N(parameter.value, 10*step).
    # Setting step=0.025 therefore gives the basin-tested sigma_start=0.25.
    return RealImag(
        Parameter.coefficient(f"{name}.x",float(x),owner=name,step=START_SIGMA/10.0),
        Parameter.coefficient(f"{name}.y",float(y),owner=name,step=START_SIGMA/10.0),
    )
coefficients={
    "sigma":free_coefficient("sigma"), "rho770":RealImag(1.0,0.0),
    "NR":free_coefficient("NR"), "f0_980":free_coefficient("f0_980"),
    "f2_1270":free_coefficient("f2_1270"), "f0_1370":free_coefficient("f0_1370"),
    "rho1450":free_coefficient("rho1450"),
}
components=[
    Resonance("sigma",(0,1),coefficients["sigma"],mass=0.4780,width=0.3240,spin=0,resonance_radius=3.0,parent_radius=3.0),
    Resonance("rho770",(0,1),coefficients["rho770"],mass=0.7693,width=0.1502,spin=1,resonance_radius=3.0,parent_radius=3.0),
    Resonance("f0_980",(0,1),coefficients["f0_980"],mass=0.9750,width=0.0440,spin=0,resonance_radius=3.0,parent_radius=3.0),
    Resonance("f2_1270",(0,1),coefficients["f2_1270"],mass=1.2750,width=0.1850,spin=2,resonance_radius=3.0,parent_radius=3.0),
    Resonance("f0_1370",(0,1),coefficients["f0_1370"],mass=1.4340,width=0.1730,spin=0,resonance_radius=3.0,parent_radius=3.0),
    Resonance("rho1450",(0,1),coefficients["rho1450"],mass=1.4650,width=0.3100,spin=1,resonance_radius=3.0,parent_radius=3.0),
    NonResonant(coefficients["NR"]),
]
model=DecayModel(channel,components)
print(f"free parameters: {len([p for p in model.parameters if not p.fixed])}")
print(f"Gaussian start width: {START_SIGMA}")


## 2. Configure the validation GenFit

The candidate pool, component amplitudes and fixed normalization matrix are cached and reused across all pseudoexperiments. `start_range=None` activates the parameter-centered Gaussian initialization; because the coefficient parameters are initialized at truth with `step=0.025`, each start is drawn with $\sigma_{start}=10\times0.025=0.25$.

The fit-quality acceptance still uses the existing EDM/error/covariance/bound criteria. In this run, $\Delta\mathrm{NLL}=\mathrm{NLL}_{fit}-\mathrm{NLL}_{truth}$ is shown explicitly as a minimization diagnostic but is **not yet used as an acceptance cut**, so we can see the effect of the start strategy in isolation.


In [ ]:
N_FITS=100
SAMPLE_SIZE=50_000
study=GenFit(
    model, n_fits=N_FITS, sample_size=SAMPLE_SIZE, truth_values=truth,
    grid_resolution=1000, pool_size=1_000_000, start_range=None,
    seed=791, pool_seed=2000, ncall=100_000, tolerance=1e-4,
    max_edm=1e-3, require_posdef_covar=True, reject_at_limit=True, verbose=1,
)
study


## 3. Run 100 pseudoexperiments


In [ ]:
result=study.run()


## 4. Convergence, acceptance, and minimization diagnostic

A fit can be `Minuit.valid` yet sit in the wrong local minimum. The single-toy study demonstrated this directly. Therefore we inspect $\Delta\mathrm{NLL}_{truth}$ for every accepted fit in addition to the standard GenFit quality criteria.


In [ ]:
result.print_summary()
accepted=result.accepted_mask
delta_truth_nll=result.nll-result.truth_nll
accepted_delta=delta_truth_nll[accepted]
print()
print(f"converged fits : {result.n_converged}/{result.n_fits} ({100*result.convergence_rate:.2f}%)")
print(f"accepted fits  : {result.n_accepted}/{result.n_fits} ({100*result.acceptance_rate:.2f}%)")
print(f"rejected fits  : {result.n_fits-result.n_accepted}/{result.n_fits}")
print(f"mean EDM (accepted)    : {np.mean(result.edm[accepted]):.4e}")
print(f"median nfcn (accepted) : {np.median(result.nfcn[accepted]):.0f}")
print(f"mean DeltaNLL_truth (accepted)   : {np.mean(accepted_delta):.6f}")
print(f"median DeltaNLL_truth (accepted) : {np.median(accepted_delta):.6f}")
print(f"max DeltaNLL_truth (accepted)    : {np.max(accepted_delta):.6f}")
print(f"accepted fits with DeltaNLL_truth > 1e-3: {np.sum(accepted_delta>1e-3)}")
print("rejection summary:", result.rejection_summary())


In [ ]:
fig,ax=plt.subplots(figsize=(7,4.8))
ax.hist(accepted_delta,bins=30,alpha=0.7)
ax.axvline(0.0,linestyle="--",label="NLL(truth)")
ax.axvline(1e-3,linestyle=":",label="diagnostic tolerance")
ax.set(xlabel=r"$\mathrm{NLL}_{fit}-\mathrm{NLL}_{truth}$",ylabel="Accepted pseudoexperiments",title="Minimization diagnostic")
ax.legend(); fig.tight_layout(); plt.show()


## 5. Primary fit-bias and pull-calibration test

Every pseudoexperiment is generated from the same injected parameter vector, while the finite sample fluctuates independently from toy to toy. Therefore individual fits are expected to differ from the truth. The closure criterion is an ensemble statement: the mean fitted value must be compatible with the truth, and the fitted uncertainties should produce pulls compatible with a standard normal distribution.

For each parameter we quote $b=\langle\hat\theta\rangle-\theta_\mathrm{true}$, the error on the ensemble mean, $b/\sigma_b$, the pull mean, and the pull width. **No MAD/outlier cut is used in this primary table.**


In [ ]:
bias_rows=genfit_bias_summary(result)
print_genfit_bias_summary(result)


## 6. Robust outlier diagnostic of accepted fits

This is a secondary diagnostic only. For each one-dimensional distribution we define $\mu_R=\mathrm{median}(x)$ and $\sigma_R=1.4826\,\mathrm{MAD}$ and flag points satisfying $|x-\mu_R|>5\sigma_R$. These points are not removed from the primary bias calculation above.


In [ ]:
OUTLIER_THRESHOLD=5.0
print_genfit_robust_summary(result,threshold=OUTLIER_THRESHOLD)
robust_rows=genfit_robust_summary(result,threshold=OUTLIER_THRESHOLD)


## 7. Parameter histograms: accepted core and robust diagnostic outliers


In [ ]:
for name in result.parameter_names:
    values=genfit_distribution(result,name)
    selection=genfit_outlier_selection(result,name,threshold=OUTLIER_THRESHOLD)
    kept=values[selection.mask]
    rejected=values[~selection.mask]
    g=genfit_robust_gaussian_fit(result,name,threshold=OUTLIER_THRESHOLD)
    fig,ax=plt.subplots(figsize=(7,4.8))
    counts,edges,_=ax.hist(kept,bins=30,alpha=0.65,label="robust core")
    if rejected.size:
        ax.hist(rejected,bins=edges,histtype="step",linewidth=2,label=f"diagnostic outliers ({rejected.size})")
    if g.valid and g.sigma>0:
        x=np.linspace(edges[0],edges[-1],500); bw=np.mean(np.diff(edges))
        y=g.n_kept*bw*np.exp(-0.5*((x-g.mean)/g.sigma)**2)/(np.sqrt(2*np.pi)*g.sigma)
        ax.plot(x,y,label="robust Gaussian fit")
    ax.axvline(truth[name],linestyle="--",label="truth")
    ax.text(0.03,0.97,f"kept = {g.n_kept}/{g.n_entries}\ndiagnostic outliers = {g.n_outliers} ({100*g.outlier_fraction:.2f}%)\nmean = {g.mean:.5f} ± {g.mean_error:.5f}\nsigma = {g.sigma:.5f} ± {g.sigma_error:.5f}",transform=ax.transAxes,va="top")
    ax.set(xlabel=name,ylabel="Pseudoexperiments",title=f"GenFit robust diagnostic: {name}")
    ax.legend(); fig.tight_layout(); plt.show()


## 8. NLL distribution with robust diagnostic


In [ ]:
name="nll"
values=genfit_distribution(result,name)
selection=genfit_outlier_selection(result,name,threshold=OUTLIER_THRESHOLD)
kept=values[selection.mask]; rejected=values[~selection.mask]
g=genfit_robust_gaussian_fit(result,name,threshold=OUTLIER_THRESHOLD)
fig,ax=plt.subplots(figsize=(7,4.8))
counts,edges,_=ax.hist(kept,bins=30,alpha=0.65,label="robust core")
if rejected.size:
    ax.hist(rejected,bins=edges,histtype="step",linewidth=2,label=f"diagnostic outliers ({rejected.size})")
if g.valid and g.sigma>0:
    x=np.linspace(edges[0],edges[-1],500); bw=np.mean(np.diff(edges))
    y=g.n_kept*bw*np.exp(-0.5*((x-g.mean)/g.sigma)**2)/(np.sqrt(2*np.pi)*g.sigma)
    ax.plot(x,y,label="robust Gaussian fit")
ax.set(xlabel="NLL",ylabel="Pseudoexperiments",title="GenFit robust NLL diagnostic")
ax.legend(); fig.tight_layout(); plt.show()


## 9. Pull distributions

For an unbiased, correctly calibrated fitter the pull ensemble should be approximately Gaussian with mean 0 and width 1. The raw pull mean and width used for the primary bias table are computed from all accepted fits. The robust curves below are visualization diagnostics only.


In [ ]:
for name in result.parameter_names:
    pulls=result.pulls(name)
    selection=robust_outlier_mask(pulls,threshold=OUTLIER_THRESHOLD)
    kept=pulls[selection.mask]; rejected=pulls[~selection.mask]
    g=robust_gaussian_fit(pulls,threshold=OUTLIER_THRESHOLD)
    fig,ax=plt.subplots(figsize=(7,4.8))
    counts,edges,_=ax.hist(kept,bins=30,alpha=0.65,label="robust core")
    if rejected.size:
        ax.hist(rejected,bins=edges,histtype="step",linewidth=2,label=f"diagnostic outliers ({rejected.size})")
    if g.valid and g.sigma>0:
        x=np.linspace(edges[0],edges[-1],500); bw=np.mean(np.diff(edges))
        y=g.n_kept*bw*np.exp(-0.5*((x-g.mean)/g.sigma)**2)/(np.sqrt(2*np.pi)*g.sigma)
        ax.plot(x,y,label="robust Gaussian fit")
    ax.axvline(0.0,linestyle="--",label="expected mean")
    ax.set(xlabel=f"pull({name})",ylabel="Pseudoexperiments",title=f"GenFit pull diagnostic: {name}")
    ax.legend(); fig.tight_layout(); plt.show()


## 10. Bias-significance overview

As a compact final check, the most important quantity for closure is the ensemble bias expressed in units of the uncertainty on the ensemble mean. Values statistically compatible with zero indicate no measurable fit bias at the tested sample size.


In [ ]:
print(f"{'parameter':18s} {'bias':>12s} {'err(mean)':>12s} {'bias/err':>10s} {'pull mean':>12s} {'pull width':>12s}")
for row in bias_rows:
    print(f"{row['name']:18s} {row['bias']:12.6g} {row['mean_error']:12.6g} {row['bias_significance']:10.3f} {row['pull_mean']:12.4f} {row['pull_width']:12.4f}")
